Name: Nicole Patricio Minchio
Assignment 4 - Data Engineering

In [2]:
import pandas as pd
df = pd.read_csv("crunchbase_odm_orgs.csv", low_memory=False)
print("Total records:", len(df))
print(df.columns)

Total records: 1127658
Index(['uuid', 'name', 'type', 'primary_role', 'cb_url', 'domain',
       'homepage_url', 'logo_url', 'facebook_url', 'twitter_url',
       'linkedin_url', 'combined_stock_symbols', 'city', 'region',
       'country_code', 'short_description'],
      dtype='str')


In [3]:
# Build a sample of 50 companies that have a non-empty short description. Keep only the columns you need: name, short description, city, region, country_code.
cols = ["name", "short_description", "city", "region", "country_code"]

has_desc = df["short_description"].notna() & (df["short_description"].str.strip() != "")

sample = (
    df.loc[has_desc, cols]
      .sample(n=50, random_state=42)
      .reset_index(drop=True)
)

print("Sample size:", len(sample))
sample.head(5)

Sample size: 50


,name,short_description,city,region,country_code
0,Soft Solutions,Soft Solutions is a web design and mobile appl...,Lahore,Punjab,PAK
1,CIF Metal,"CIF Metal is specialize in heavyweight, mid to...",Thetford Mines,Quebec,CAN
2,LiveCA LLP,LiveCA is a full-service online accounting fir...,Toronto,Ontario,CAN
3,Tikona Infinet Limited,Engaged in building the next generation wirele...,Bhandup,Maharashtra,IND
4,Incite Marketing Planning,Incite Marketing Planning operates as a strat...,London,England,GBR


I kept only rows where `short_description` is not empty (not NaN and not just spaces), then took 50 random rows. I set `random_state=42` so I get the same 50 companies every time I rerun it. I kept the five columns I need. 

The `value_counts()`  below shows which countries ended up in the sample. 

In [4]:
print(sample["country_code"].value_counts())

country_code
USA    21
CAN     6
IND     2
GBR     2
CHN     2
FIN     2
NGA     2
PAK     1
JPN     1
FRA     1
ECU     1
NLD     1
MYS     1
UGA     1
POL     1
Name: count, dtype: int64


In [5]:
# Task 1: Embed all 50 descriptions with all-MiniLM-L6-v2 and write your own cosine() function using numpy.
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # avoids a common Mac/conda crash; must come before torch loads

import gc
try:
    del df           
except NameError:
    pass
gc.collect()

from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

descriptions = sample["short_description"].tolist()
doc_vectors = model.encode(descriptions, batch_size=8)  

print("Embedding matrix shape:", doc_vectors.shape)   

/opt/anaconda3/envs/apan-5400/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1118 15:48:11.960000 4126 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 8731.07it/s]


Embedding matrix shape: (50, 384)


I turned each of the 50 descriptions into a vector with `all-MiniLM-L6-v2`. The result has shape (50, 384): 50 companies, 384 numbers each.

Things I had to deal with here:
- `sentence-transformers` was not installed in my `apan-5400` environment, so I got a `ModuleNotFoundError` and had to install it with `%pip install sentence-transformers`.
- The first time I ran the embedding, my kernel died. I think it ran out of memory. To fix it, I delete `df` before loading the model (I only need `sample` from here on) and use a small `batch_size=8`.
- `KMP_DUPLICATE_LIB_OK` is set because Mac with conda can crash when two copies of a math library load. It has to be set before PyTorch loads.
- The `tqdm` / `IProgress` warning only affects the progress bar. It is not an error.
- After the kernel restarted I got `NameError: name 'sample' is not defined`, because restarting clears all variables. I reran the earlier cells in order.

In [6]:
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print(cosine(doc_vectors[0], doc_vectors[0]))  
print(cosine(doc_vectors[0], doc_vectors[1]))   

1.0
-0.03484509885311127


In [7]:
def search(query, top_k=5):
    query_vector = model.encode(query)
    scores = [cosine(query_vector, v) for v in doc_vectors]
    ranked = sorted(zip(scores, sample["name"], sample["short_description"]), reverse=True)

    print(f"Query: {query!r}")
    for score, name, desc in ranked[:top_k]:
        print(f"  {score:.3f}  {name}: {desc[:90]}")
    print()


`cosine(a, b)` is the dot product of two vectors divided by the product of their lengths. Two checks: a vector compared with itself gives 1.0, and two unrelated companies (Soft Solutions and CIF Metal) give about -0.03.

`search()` embeds the query, scores all 50 companies with `cosine()`, sorts by score, and prints the top 5.

In [8]:
for i, row in sample.iterrows():
    print(i, row["name"], "|", row["short_description"][:100])

0 Soft Solutions | Soft Solutions is a web design and mobile application development company.
1 CIF Metal | CIF Metal is specialize in heavyweight, mid to large volume aluminum casting, using permanent and se
2 LiveCA LLP | LiveCA is a full-service online accounting firm focused on dynamic Canadian entrepreneurs and small 
3 Tikona Infinet Limited | Engaged in building the next generation wireless broadband services for home and enterprise customer
4 Incite Marketing Planning |  Incite Marketing Planning operates as a strategic research consultancy.
5 REFLX Labs | REFLX Labs, Inc. is a Seattle-based technology firm dedicated to the development and distribution of
6 BCS Overland Park / Leawood Kansas | BCS Overland Park / Leawood Kansas is a job hunting company for executives, offering personalized ca
7 Berkshire Welco | Berkshire Welco is focuses on the cultivation, manufacturing, and retailing recreational marijuana.
8 Edsby | Developer of an innovative digital learning and data softw

In [9]:
# Search 1: a topic some companies work on, phrased in different words than their descriptions
search("helping students and professionals build new skills through instruction and coaching")

# Search 2: a topic none of the companies cover
search("medieval Portuguese poetry")

Query: 'helping students and professionals build new skills through instruction and coaching'
  0.362  Edsby: Developer of an innovative digital learning and data software platform for K-12 school dis
  0.312  The Mitchell Group: The Mitchell Group provides practical solutions.
  0.292  Inrol: Inrol Provides High-stakes communication training and 'real-life' simulations.
  0.254  Big Clever Learning: Big Clever Learning, is a private-equity backed business that focuses on acquiring, develo
  0.241  Wondough: Making Online Experience Rewarding 

Query: 'medieval Portuguese poetry'
  0.200  Gestoria Zaragoza: Gestoría Zaragoza para agilizar pago de impuestos. Trabajamos de manera rápida y discreta.
  0.189  Teemonation: The Balkan League of Legends community.
  0.133  Remet ehitus: Remet ehitus on andnud suure panuse Tallinna, ümbritsevate valdade ja kogu Eesti Vabariigi
  0.132  Yasta: Next Super Mobile App in Latin America
  0.121  C2M LLC: C2M offers a cloud-based platform for automat

Search 1 is about education and training. I phrased it as "helping students and professionals build new skills through instruction and coaching", which avoids the words "learning", "training", "school districts".

Search 2 is "medieval Portuguese poetry", which none of the companies cover. It still returned 5 companies, but the top score was only 0.200. 

In [5]:
# Task 2: Call an LLM through the Gemini API
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()   

gemini_client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.environ["GEMINI_API_KEY"],
)

MODEL = "gemini-3.1-flash-lite" 
print("Client ready, model:", MODEL)

Client ready, model: gemini-3.1-flash-lite


I used the `openai` library with the Gemini base URL, as in the class notebook. The key is read from my `.env` file and not typed into the notebook. The model is `gemini-3.1-flash-lite`.

`openai` wasn't installed in my environment either, so I installed it with `%pip install openai`.

In [21]:
# Send one company's short description to the model and ask it to classify that company into one of: fintech, health, saas, hardware, other. 
company = sample.iloc[27]

prompt = (
    "Classify this company into exactly one of these categories: "
    "fintech, health, saas, hardware, other.\n"
    "Reply with only the category name, in lowercase.\n\n"
    f"Company: {company['name']}\n"
    f"Description: {company['short_description']}"
)

response = gemini_client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
    temperature=0,  
)

print("Call completed for:", company["name"])

Call completed for: e-Revenue Gateway


I sent one company (e-Revenue Gateway, row 27) and asked for exactly one of: fintech, health, saas, hardware, other. I used `temperature=0` so the same input gives the same answer each time.

In [12]:
# Print the model's answer and response.usage
print("Company:", company["name"])
print("Description:", company["short_description"])
print("Model answer:", response.choices[0].message.content)
print("Usage:", response.usage)

Company: e-Revenue Gateway
Description: e-Revenue Gateway (e-RG) is a payment processing and electronic transaction switching company.
Model answer: fintech
Usage: CompletionUsage(completion_tokens=2, prompt_tokens=63, total_tokens=65, completion_tokens_details=None, prompt_tokens_details=None)


The model answered `fintech`, which fits a payment processing company. `response.usage` shows 63 prompt tokens, 2 completion tokens, and 65 total. The completion is small because I asked for the category name only.

In [6]:
# Task 3: Write one tool function, find_companies(country_code, limit), that filters your 50-row sample by country and returns at most limit companies. Then write its tool description.
import json

def find_companies(country_code, limit=5):
    """Return at most `limit` companies from the 50-row sample in the given country."""
    matches = sample[sample["country_code"].str.upper() == str(country_code).upper()]
    matches = matches.head(int(limit)).fillna("")   
    return matches[["name", "short_description", "city", "region", "country_code"]].to_dict("records")

print(find_companies("JPN", 5))
print(find_companies("CAN", 3))

[{'name': 'Heian Ceremony Service', 'short_description': 'Heian Ceremony Service is a platform dealing with wedding & funeral business & other social business that includes photography to dressing.', 'city': 'Hiratsuka', 'region': 'Kanagawa', 'country_code': 'JPN'}]
[{'name': 'CIF Metal', 'short_description': 'CIF Metal is specialize in heavyweight, mid to large volume aluminum casting, using permanent and semi-permanent molds.', 'city': 'Thetford Mines', 'region': 'Quebec', 'country_code': 'CAN'}, {'name': 'LiveCA LLP', 'short_description': 'LiveCA is a full-service online accounting firm focused on dynamic Canadian entrepreneurs and small business owners.', 'city': 'Toronto', 'region': 'Ontario', 'country_code': 'CAN'}, {'name': 'Edsby', 'short_description': 'Developer of an innovative digital learning and data software platform for K-12 school districts, states/provinces and countries.', 'city': 'Richmond Hill', 'region': 'Ontario', 'country_code': 'CAN'}]


`find_companies(country_code, limit)` filters the 50-row sample by country and returns at most `limit` companies. I tested it directly with JPN (1 result) and CAN with limit 3 (3 results) before involving the model. `fillna("")` replaces missing values so the result can be turned into JSON.

In [19]:
available_codes = sorted(sample["country_code"].dropna().unique().tolist())
print("Codes in sample:", available_codes)

tools = [{
    "type": "function",
    "function": {
        "name": "find_companies",
        "description": (
            "List companies from the Crunchbase sample that are located in a given country. "
            "Use this when the user asks which companies are in, or based in, a country. "
            "Country codes are 3-letter ISO codes (e.g. USA, JPN, CAN, GBR). "
            f"Codes present in the data: {', '.join(available_codes)}."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "country_code": {
                    "type": "string",
                    "description": "3-letter country code, e.g. JPN for Japan, CAN for Canada",
                },
                "limit": {
                    "type": "integer",
                    "description": "Maximum number of companies to return (default 5)",
                },
            },
            "required": ["country_code"],
        },
    },
}]

Codes in sample: ['CAN', 'CHN', 'ECU', 'FIN', 'FRA', 'GBR', 'IND', 'JPN', 'MYS', 'NGA', 'NLD', 'PAK', 'POL', 'UGA', 'USA']


The tool description tells the model when to use the function and what the arguments mean. I included the list of country codes that exist in my sample so the model can turn "Japan" into JPN.

In [20]:
# Run the agent round-trip: send the question along with your tool, and if the model asks for the tool, run it, send the result back, and print the final answer.
def run_agent(question):
    messages = [{"role": "user", "content": question}]
    response = gemini_client.chat.completions.create(
        model=MODEL, messages=messages, tools=tools,
    )
    msg = response.choices[0].message
    if msg.tool_calls:
        call = msg.tool_calls[0]
        args = json.loads(call.function.arguments)   

        if call.function.name == "find_companies":
            result = find_companies(**args)
        else:
            result = {"error": f"unknown tool {call.function.name}"}
        messages.append(msg)   
        messages.append({
            "role": "tool",
            "tool_call_id": call.id,
            "content": json.dumps(result),
        })
        final = gemini_client.chat.completions.create(model=MODEL, messages=messages)
        print("Final answer:", final.choices[0].message.content)
    else:
        print("No tool requested. Answer:", msg.content)

run_agent("What companies do you have in Japan?")

Final answer: I have information about one company in Japan in my database:

*   **Heian Ceremony Service**
    *   **Description:** A company that provides wedding and funeral services, as well as related social services such as photography and dressing.
    *   **Location:** Hiratsuka, Kanagawa, Japan.


In [16]:
# Do this for two different questions that require different arguments, for example "What companies do you have in Japan?" and "Show me 3 companies in Canada."
run_agent("What companies do you have in Japan?")
print()
run_agent("Show me 3 companies in Canada.")

Final answer: In my current database, I have information on the following company in Japan:

*   **Heian Ceremony Service:** Based in Hiratsuka, Kanagawa, this company operates in the wedding and funeral services sector, which also includes photography and dressing services. 

If you are looking for information on a specific industry or a broader range of companies in Japan, please let me know, and I can refine my search or provide more details!

Final answer: Here are three companies based in Canada:

1.  **CIF Metal** (Thetford Mines, Quebec): Specializes in heavyweight, mid-to-large volume aluminum casting using permanent and semi-permanent molds.
2.  **LiveCA LLP** (Toronto, Ontario): A full-service online accounting firm focused on supporting Canadian entrepreneurs and small business owners.
3.  **Edsby** (Richmond Hill, Ontario): A technology company that develops a digital learning and data platform designed for K-12 school districts, provinces, and countries.


In [17]:
# For each question, print the tool name, the arguments the model chose, the tool result, and the final answer.
def run_agent(question):
    print("=" * 70)
    print("QUESTION:", question)

    messages = [{"role": "user", "content": question}]

    response = gemini_client.chat.completions.create(
        model=MODEL, messages=messages, tools=tools,
    )
    msg = response.choices[0].message

    if msg.tool_calls:
        call = msg.tool_calls[0]
        args = json.loads(call.function.arguments)  

        if call.function.name == "find_companies":
            result = find_companies(**args)
        else:
            result = {"error": f"unknown tool {call.function.name}"}
        print("tool name  :", call.function.name)
        print("arguments  :", args)
        print("tool result:", json.dumps(result, indent=2))
        messages.append(msg)
        messages.append({
            "role": "tool",
            "tool_call_id": call.id,
            "content": json.dumps(result),
        })
        final = gemini_client.chat.completions.create(model=MODEL, messages=messages)
        print("\nFINAL ANSWER:", final.choices[0].message.content)
    else:
        print("No tool requested. Answer:", msg.content)

run_agent("What companies do you have in Japan?")
print()
run_agent("Show me 3 companies in Canada.")

QUESTION: What companies do you have in Japan?
tool name  : find_companies
arguments  : {'country_code': 'JPN'}
tool result: [
  {
    "name": "Heian Ceremony Service",
    "short_description": "Heian Ceremony Service is a platform dealing with wedding & funeral business & other social business that includes photography to dressing.",
    "city": "Hiratsuka",
    "region": "Kanagawa",
    "country_code": "JPN"
  }
]

FINAL ANSWER: The information I have access to currently includes **Heian Ceremony Service**, which is based in Hiratsuka, Kanagawa. This company operates in the wedding and funeral industry, and also provides related services such as photography and garment dressing.

QUESTION: Show me 3 companies in Canada.
tool name  : find_companies
arguments  : {'limit': 3, 'country_code': 'CAN'}
tool result: [
  {
    "name": "CIF Metal",
    "short_description": "CIF Metal is specialize in heavyweight, mid to large volume aluminum casting, using permanent and semi-permanent molds.",

The model can’t query my data itself.It only sees the results I send back. I send the question plus the tool description. If the model asks for the tool, my code runs `find_companies` with the arguments it chose, and I send the result back so the model can write the final answer.

For both questions I print the tool name, the arguments the model chose, the tool result, and the final answer.
"What companies do you have in Japan?" → `{'country_code': 'JPN'}`, 1 company.
"Show me 3 companies in Canada." → `{'country_code': 'CAN', 'limit': 3}`, 3 companies.

The model picked different arguments for each question, and in the Canada question it pulled the number 3 out of the sentence itself.